<a href="https://www.kaggle.com/code/elmiraviktorovich/titanic?scriptVersionId=315509427" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Titanic: Machine Learning from Disaster

## 1. Введение и постановка задачи
Цель проекта — предсказать выживаемость пассажиров Титаника на основе имеющихся признаков (пол, возраст, класс билета и др.). Это задача бинарной классификации.

В данной работе мы пройдем полный цикл Data Science:
- Первичный анализ и очистка данных
- Разведочный анализ (EDA) и визуализация
- Feature Engineering (создание новых признаков)
- Построение и оценка моделей машинного обучения

# 2. Первичный анализ данных

## 2.1 Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import random
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import warnings
import missingno as msno
warnings.filterwarnings('ignore')

# sklearn базовые
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

In [ ]:
# Настройка цветовой палитры
COLORS = {
    'survived': '#20A39E',  # зелёный - выжил
    'died': '#EF5B5B',      # красный - погиб
    'male': '#2E86AB',      # синий - мужчина
    'female': '#A23B72',    # фиолетовый - женщина
    'class_1': '#2E86AB',   # синий
    'class_2': '#FFBA08',   # жёлтый
    'class_3': '#A23B72',   # фиолетовый
    'embarked_S': '#2E86AB',
    'embarked_C': '#20A39E',
    'embarked_Q': '#FFBA08'
}

# Базовая палитра seaborn
sns.set_palette('Set2')
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Константы проекта
TARGET = 'Survived'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

## 2.2 Предварительный осмотр структуры данных



Для ускорения EDA подготовлен набор функций, агрегирующих ключевую информацию о датасете:

| Категория | Функция | Назначение |
|:---|:---|:---|
| **Пропуски** | `missing_data()` | Детальная статистика по пропущенным значениям |
| | `compare_train_test()` | Сравнение процента пропусков в train и test |
| **Значения** | `most_frequent_values()` | Самые частые значения в каждой колонке |
| | `unique_values_stats()` | Статистика по уникальным значениям и кардинальность |
| **Визуал** | `display_side_by_side()` | Отображает два DataFrame рядом друг с другом |

Для визуализации данных подготовлен набор функций:

| Категория | Функция | Назначение |
|:---|:---|:---|
| **Пропуски** | `missingno_matrix()` | Матрица расположения пропусков |
| | `missingno_bar()` | Столбчатая диаграмма |
| **EDA** |`plot_categorical()` | Анализ категориальных признаков |
| | `plot_numerical()` | Анализ числовых признаков |
| | `plot_correlation_matrix()` | Корреляционная матрица |

In [ ]:
from data_quality import missing_data, compare_train_test, most_frequent_values, unique_values_stats, display_side_by_side
from visualization import missingno_matrix, missingno_bar, plot_categorical, plot_numerical, plot_correlation_matrix

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

## 2.3 Приведение типов данных

In [ ]:
# Возраст должен быть float, а не object
train_df['Age'] = train_df['Age'].astype(float)
test_df['Age'] = test_df['Age'].astype(float)

# Категориальные признаки лучше сделать типом 'category'
categorical_cols = ['Sex', 'Embarked', 'Pclass']
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

if 'Survived' in train_df.columns:
    train_df['Survived'] = train_df['Survived'].astype(int)

# 3. Очистка данных (Data Cleaning)

## 3.1 Обработка пропусков

In [ ]:
# Детальная статистика по пропускам
display_side_by_side(
    missing_data(train_df),
    missing_data(test_df),
    "Пропуски ДО обработки - TRAIN",
    "Пропуски ДО обработки - TEST"
)

In [ ]:
# Сравнение пропусков Train vs Test
print("=== СРАВНЕНИЕ ПРОПУСКОВ TRAIN vs TEST ===")
display(compare_train_test(train_df, test_df))

In [ ]:
display_side_by_side(
    most_frequent_values(train_df),
    most_frequent_values(test_df),
    "Самые частые значения - TRAIN",
    "Самые частые значения - TEST"
)

In [ ]:
# Уникальные значения
print("=== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ В TRAIN ===")
display(unique_values_stats(train_df))

print("=== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ В TEST ===")
display(unique_values_stats(test_df))

In [ ]:
# Сравнение пропусков train и test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

msno.bar(train_df,color=(32/255, 163/255, 158/255), ax=axes[0])
axes[0].set_title('Train - Полнота колонок', fontsize=12, fontweight='bold')

msno.matrix(test_df,color=(32/255, 163/255, 158/255), ax=axes[1])
axes[1].set_title('Test - Матрица пропусков', fontsize=12, fontweight='bold')

plt.suptitle('Сравнение паттернов пропусков: Train vs Test',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.1.1 Заполнение Embarked (порт посадки)

In [ ]:
# Всего 2 пропуска в train. Заполняем самым частым значением (модой)
most_common_embarked = train_df['Embarked'].mode()[0]
print(f"\nЗаполняем Embarked: мода = '{most_common_embarked}'")

train_df['Embarked'] = train_df['Embarked'].fillna(most_common_embarked)
test_df['Embarked'] = test_df['Embarked'].fillna(most_common_embarked)

### 3.1.2 Заполнение Fare (только в test, 1 пропуск)

In [ ]:
if test_df['Fare'].isnull().sum() > 0:
    fare_median = test_df['Fare'].median()
    print(f"\nЗаполняем Fare в test: медиана = {fare_median:.2f}")
    test_df['Fare'] = test_df['Fare'].fillna(fare_median)

### 3.1.3 Заполнение Age (много пропусков)

In [ ]:
# Сначала извлечем Title из имени
def extract_title(name):
    title = name.split(',')[1].split('.')[0].strip()
    # Группируем редкие титулы
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                   'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    if title in rare_titles:
        return 'Rare'
    elif title in ['Mlle', 'Ms']:
        return 'Miss'
    elif title == 'Mme':
        return 'Mrs'
    else:
        return title

# Добавляем временный признак Title для заполнения
train_df['Title_temp'] = train_df['Name'].apply(extract_title)
test_df['Title_temp'] = test_df['Name'].apply(extract_title)

# Заполняем Age медианой по группе (Title + Pclass)
for df in [train_df, test_df]:
    for title in df['Title_temp'].unique():
        for pclass in df['Pclass'].unique():
            mask = (df['Title_temp'] == title) & (df['Pclass'] == pclass)
            median_age = df.loc[mask, 'Age'].median()
            if pd.notna(median_age):
                df.loc[mask & df['Age'].isna(), 'Age'] = median_age

# Оставшиеся пропуски заполняем общей медианой
overall_median_age = train_df['Age'].median()
train_df['Age'] = train_df['Age'].fillna(overall_median_age)
test_df['Age'] = test_df['Age'].fillna(overall_median_age)

# Удаляем временный признак
train_df.drop('Title_temp', axis=1, inplace=True)
test_df.drop('Title_temp', axis=1, inplace=True)

print(f"   Age заполнен. Медиана: {overall_median_age:.1f}")

In [ ]:
display_side_by_side(
    missing_data(train_df),
    missing_data(test_df),
    "Пропуски ПОСЛЕ обработки - TRAIN",
    "Пропуски ПОСЛЕ обработки - TEST"
)

**Cabin: пропуски НЕ заполняем, создадим отдельные признаки позже**

## 3.2 Обработка выбросов

In [ ]:
fare_upper = train_df['Fare'].quantile(0.99)
print(f"\nОграничиваем Fare значением {fare_upper:.2f} (99-й перцентиль)")
train_df['Fare'] = np.where(train_df['Fare'] > fare_upper, fare_upper, train_df['Fare'])
test_df['Fare'] = np.where(test_df['Fare'] > fare_upper, fare_upper, test_df['Fare'])

# 4. Разведочный анализ данных (EDA)

## 4.1 Анализ целевой переменной (Survived)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Countplot с использованием настроенного словаря
sns.countplot(data=train_df, x='Survived', hue='Survived', ax=axes[0],
              palette=[COLORS['died'], COLORS['survived']], legend=False)
axes[0].set_title('Количество выживших и погибших', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Survived')
axes[0].set_ylabel('Количество пассажиров')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Погиб (0)', 'Выжил (1)'])

# Добавляем значения на столбцы
for i, v in enumerate(train_df['Survived'].value_counts().sort_index()):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
survived_counts = train_df['Survived'].value_counts()
axes[1].pie(survived_counts.values,
            labels=['Погиб', 'Выжил'],
            autopct='%1.1f%%',
            colors=[COLORS['died'], COLORS['survived']],
            startangle=90,
            explode=(0.02, 0.02),
            textprops={'fontsize': 12})
axes[1].set_title('Процентное соотношение', fontsize=12, fontweight='bold')

plt.suptitle(' Целевая переменная: Survived', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Статистика
print(f"Статистика:")
print(f"   • Выжило:  {train_df['Survived'].sum()} пассажиров ({train_df['Survived'].mean()*100:.1f}%)")
print(f"   • Погибло: {len(train_df) - train_df['Survived'].sum()} пассажиров ({(1-train_df['Survived'].mean())*100:.1f}%)")
print(f"   • Baseline accuracy (все погибли): {(1-train_df['Survived'].mean())*100:.1f}%")


## 4.2 Анализ категориальных признаков

Для начала анализа объединим обучающие и тестовые данные в один фрейм данных.

In [ ]:
all_df = pd.concat([train_df, test_df], axis=0)
all_df["set"] = "train"
all_df.loc[all_df.Survived.isna(), "set"] = "test"

In [ ]:
all_df.head()

In [ ]:

cat_features = ['Sex', 'Pclass', 'Embarked']

for col in cat_features:
    # Визуализация с помощью утилиты
    plot_categorical(train_df, col, target='Survived', figsize=(14, 5))

    # Текстовый вывод
    print(f"\n {col}:")
    print(f"   Пропусков в all_df: {all_df[col].isnull().sum()} ({all_df[col].isnull().mean()*100:.1f}%)")
    print(f"   Выживаемость по группам:")
    for val in train_df[col].dropna().unique():
        rate = train_df[train_df[col] == val]['Survived'].mean()
        count = train_df[train_df[col] == val].shape[0]
        print(f"      • {val}: {rate:.1%} (n={count})")

### Выводы по категориальным признакам:

**Sex (Пол):**
- Женщины выживали значительно чаще (74.2% vs 18.9%)
- Самый сильный предиктор выживаемости

**Pclass (Класс билета):**
- Чем выше класс, тем выше выживаемость
- 1 класс: 63.0%, 2 класс: 47.3%, 3 класс: 24.2%
- Большинство пассажиров — 3 класс (~55%)

**Embarked (Порт посадки):**
- Большинство село в Southampton (S) — ~72%
- Выживаемость выше у пассажиров из Cherbourg (C) — 55.4%
- Связано с классом: в C больше пассажиров 1 класса

## 4.3 Анализ числовых признаков

In [ ]:
# Числовые признаки для анализа
num_features = ['Age', 'Fare', 'SibSp', 'Parch']

for col in num_features:
    # готовую функцию
    plot_numerical(train_df, col, target='Survived', figsize=(14, 5))

    # Дополнительно выводим статистику
    print(f"\n {col}:")
    print(f"   Пропусков в all_df: {all_df[col].isnull().sum()} ({all_df[col].isnull().mean()*100:.1f}%)")
    print(f"   Статистика по train:")
    print(f"      • Среднее (выжившие): {train_df[train_df['Survived'] == 1][col].mean():.2f}")
    print(f"      • Среднее (погибшие): {train_df[train_df['Survived'] == 0][col].mean():.2f}")
    print(f"      • Медиана (выжившие): {train_df[train_df['Survived'] == 1][col].median():.2f}")
    print(f"      • Медиана (погибшие): {train_df[train_df['Survived'] == 0][col].median():.2f}")

In [ ]:
# Создаём временный признак FamilySize
train_df_temp = train_df.copy()
train_df_temp['FamilySize'] = train_df_temp['SibSp'] + train_df_temp['Parch'] + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FamilySize vs Survived
family_survival = train_df_temp.groupby('FamilySize')['Survived'].mean()
axes[0].bar(family_survival.index, family_survival.values,
            color=['#EF5B5B' if x < train_df['Survived'].mean() else '#20A39E' for x in family_survival.values])
axes[0].axhline(y=train_df['Survived'].mean(), color='black', linestyle='--', alpha=0.5, label='Baseline')
axes[0].set_title('Выживаемость по размеру семьи', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Размер семьи')
axes[0].set_ylabel('Survival Rate')
axes[0].legend()

# Распределение размеров семей
family_counts = train_df_temp['FamilySize'].value_counts().sort_index()
axes[1].bar(family_counts.index, family_counts.values, color='#2E86AB', alpha=0.7)
axes[1].set_title('Распределение размеров семей', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Размер семьи')
axes[1].set_ylabel('Количество')

plt.suptitle(' Анализ семейных связей', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n Семейные признаки (SibSp + Parch):")
print(f"   Одиночек (FamilySize=1): {len(train_df_temp[train_df_temp['FamilySize'] == 1])}")
print(f"   Выживаемость одиночек: {train_df_temp[train_df_temp['FamilySize'] == 1]['Survived'].mean():.1%}")
print(f"   Выживаемость семейных (2-4 чел): {train_df_temp[(train_df_temp['FamilySize'] >= 2) & (train_df_temp['FamilySize'] <= 4)]['Survived'].mean():.1%}")
print(f"   Выживаемость больших семей (5+ чел): {train_df_temp[train_df_temp['FamilySize'] >= 5]['Survived'].mean():.1%}")

### 4.3.1 Совместное влияние размера семьи и класса билета

Рассмотрев `FamilySize` отдельно, интересно посмотреть, как он взаимодействует
с ключевым социальным признаком — **классом билета (`Pclass`)**.

Ниже представлен трехмерный график, показывающий распределение пассажиров
в координатах `FamilySize` × `Pclass`, где высота столбцов соответствует
доле пассажиров, а цвет — выживаемости.

In [ ]:
# Преобразуем Pclass в числовой тип
train_df_temp['Pclass'] = train_df_temp['Pclass'].astype(int)

# Определяем цвета для выживших и погибших
color_list = [COLORS['died'], COLORS['survived']]  # красный и зеленый

# Создаем фигуру
fig = plt.figure(figsize=(14, 10))

# ===== ГРАФИК 1: 3D KDE (вид сверху-сбоку) =====
ax1 = fig.add_subplot(121, projection='3d')

# Строим KDE для выживших и погибших
# Примечание: seaborn в 3D не умеет hue, поэтому строим отдельно для каждого класса
for survived_val, color, label in [(0, COLORS['died'], 'Погибшие'),
                                     (1, COLORS['survived'], 'Выжившие')]:
    subset = train_df_temp[train_df_temp['Survived'] == survived_val]
    # Используем hist вместо kde для 3D, так как kde в 3D работает иначе
    hist, xedges, yedges = np.histogram2d(subset['FamilySize'], subset['Pclass'],
                                           bins=[8, 3],
                                           range=[[0.5, 8.5], [0.5, 3.5]])

    # Координаты центров столбцов
    xpos, ypos = np.meshgrid(xedges[:-1] + 0.5, yedges[:-1] + 0.5, indexing="ij")
    xpos = xpos.ravel()
    ypos = ypos.ravel()
    zpos = np.zeros_like(xpos)

    # Высоты столбцов (нормированные)
    dz = hist.ravel() / len(subset)

    # Рисуем 3D столбцы
    ax1.bar3d(xpos, ypos, zpos, dx=0.8, dy=0.8, dz=dz,
              color=color, alpha=0.7, label=label)

ax1.set_xlabel('Размер семьи (FamilySize)')
ax1.set_ylabel('Класс билета (Pclass)')
ax1.set_zlabel('Доля пассажиров')
ax1.set_title('3D распределение: FamilySize vs Pclass', fontsize=12, fontweight='bold')
ax1.legend()

# Настройка меток осей
ax1.set_xticks([1, 2, 3, 4, 5, 6, 7, 8])
ax1.set_yticks([1, 2, 3])

# ===== ГРАФИК 2: 2D KDE с hue (более четкий) =====
ax2 = fig.add_subplot(122)

# 2D KDE контурный график (более читаемый для категорий)
sns.kdeplot(data=train_df_temp, x='FamilySize', y='Pclass',
            hue='Survived', palette=color_list,
            fill=True, alpha=0.3, ax=ax2)

# Добавляем точки выживших и погибших для наглядности
sns.scatterplot(data=train_df_temp, x='FamilySize', y='Pclass',
                hue='Survived', palette=color_list,
                alpha=0.4, s=30, ax=ax2, legend=False)

ax2.set_xlabel('Размер семьи (FamilySize)')
ax2.set_ylabel('Класс билета (Pclass)')
ax2.set_title('2D KDE контуры: FamilySize vs Pclass', fontsize=12, fontweight='bold')
ax2.set_yticks([1, 2, 3])
ax2.legend(title='Survived', labels=['Погибшие', 'Выжившие'])

plt.suptitle('Выживаемость в зависимости от размера семьи и класса билета',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Выводы по числовым признакам:

**Age (Возраст):**
- Дети (0-12 лет) выживали значительно чаще
- Погибшие в среднем старше (29.8 vs 28.06 лет)

**Fare (Цена билета):**
- Выжившие платили в среднем больше (45.93 vs 22.07)
- Рекомендуется логарифмировать для моделей

**SibSp + Parch (Семья):**
- Одиночки выживали хуже (30.4%)
- Семьи из 2-4 человек — лучше всех (57.9%)
- Большие семьи (5+) — хуже всех (16.1%)
- Стоит создать признак FamilySize и IsAlone

## 4.4 Корреляционный анализ

In [ ]:
# Используем утилиту
plot_correlation_matrix(train_df, figsize=(12, 10))

# Выводы
print(" Корреляции с Survived:")
correlations = train_df.corr(numeric_only=True)['Survived'].sort_values(ascending=False)
for col, corr in correlations.items():
    if col != 'Survived':
        direction = "↑" if corr > 0 else "↓"
        print(f"   • {col}: {corr:+.3f} {direction}")

## 4.5 Выводы по EDA

На основе анализа мы определили, что для улучшения модели стоит создать следующие признаки:
- Title (титул из имени)
- FamilySize (размер семьи)
- IsAlone (одинок ли пассажир)
- CabinLetter (первая буква каюты)
- AgeGroup (группы возраста)

Переходим к созданию этих признаков.

# 5. Feature Engineering (Создание признаков)

## 5.1 Создание признаков на основе Name (Title)

In [ ]:
def extract_title(name):
    title = name.split(',')[1].split('.')[0].strip()
    # Группируем редкие титулы
    if title in ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']:
        return 'Rare'
    elif title in ['Mlle', 'Ms']:
        return 'Miss'
    elif title == 'Mme':
        return 'Mrs'
    else:
        return title

train_df['Title'] = train_df['Name'].apply(extract_title)
test_df['Title'] = test_df['Name'].apply(extract_title)

In [ ]:
from wordcloud import WordCloud, STOPWORDS
stopwords = set(STOPWORDS)

def show_wordcloud(data, mask=None, title=""):
    text = " ".join(t for t in data.dropna())
    stopwords = set(STOPWORDS)
    stopwords.update(["t", "co", "https", "amp", "U", "Comment", "text", "attr", "object"])
    wordcloud = WordCloud(stopwords=stopwords, scale=4, max_font_size=50, max_words=500,mask=mask, background_color="white",
                         colormap='Set2').generate(text)
    fig = plt.figure(1, figsize=(12, 12))
    plt.axis('off')
    fig.suptitle(title, fontsize=14)
    fig.subplots_adjust(top=2.3)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.show()

In [ ]:
# Создаем фигуру с двумя подграфиками
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Облако для выживших
text_survived = " ".join(t for t in train_df[train_df['Survived'] == 1]['Name'].dropna())
stopwords = set(STOPWORDS)
stopwords.update(["t", "co", "https", "amp", "U", "Comment", "text", "attr", "object"])
wordcloud_survived = WordCloud(stopwords=stopwords, scale=4, max_font_size=50, max_words=300,
                                background_color="white", colormap='Greens').generate(text_survived)
axes[0].imshow(wordcloud_survived, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('ВЫЖИВШИЕ', fontsize=16, fontweight='bold', color='green')

# Облако для погибших
text_died = " ".join(t for t in train_df[train_df['Survived'] == 0]['Name'].dropna())
wordcloud_died = WordCloud(stopwords=stopwords, scale=4, max_font_size=50, max_words=300,
                            background_color="white", colormap='Reds').generate(text_died)
axes[1].imshow(wordcloud_died, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('ПОГИБШИЕ', fontsize=16, fontweight='bold', color='red')

plt.suptitle('Сравнение облаков слов: Выжившие vs Погибшие', fontsize=18, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 5.2 Создание семейных признаков (FamilySize, IsAlone)

In [ ]:
# Суммируем siblings + parents + сам пассажир
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1

In [ ]:
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)
test_df['IsAlone'] = (test_df['FamilySize'] == 1).astype(int)

## 5.3 Создание признаков на основе Cabin

In [ ]:
train_df['HasCabin'] = train_df['Cabin'].notna().astype(int)
test_df['HasCabin'] = test_df['Cabin'].notna().astype(int)

In [ ]:
def get_deck(text):
    try:
        return text[0]
    except Exception as ex:
        return "Unknown"

In [ ]:
all_df["Deck"] = all_df["Cabin"].apply(lambda x: get_deck(x))

Создим сводную таблицу, которая показывает, как пассажиры разных классов (Pclass) распределены по палубам (Deck)

In [ ]:
np.transpose(pd.crosstab(all_df['Deck'], all_df['Pclass']))

## 5.4 Создание признаков на основе Ticket

In [ ]:
# Создаем ВРЕМЕННЫЙ объединенный DataFrame ТОЛЬКО для расчета
temp_combined = pd.concat([train_df, test_df], axis=0)

# Рассчитываем размер группы по билету
ticket_counts = temp_combined.groupby('Ticket')['PassengerId'].transform('count')

# Добавляем колонку в ОРИГИНАЛЬНЫЕ train_df и test_df
train_df['TicketGroupSize'] = ticket_counts.iloc[:len(train_df)].values
test_df['TicketGroupSize'] = ticket_counts.iloc[len(train_df):].values

# Удаляем временный DataFrame (освобождаем память)
del temp_combined

print(" TicketGroupSize добавлен в train_df и test_df")
print(f"   train_df shape: {train_df.shape}")
print(f"   test_df shape:  {test_df.shape}")

In [ ]:
train_df['FarePerPerson'] = train_df['Fare'] / train_df['FamilySize']
test_df['FarePerPerson'] = test_df['Fare'] / test_df['FamilySize']

# Заполняем бесконечности (если FamilySize = 0)
train_df['FarePerPerson'].replace([np.inf, -np.inf], train_df['Fare'].median(), inplace=True)
test_df['FarePerPerson'].replace([np.inf, -np.inf], test_df['Fare'].median(), inplace=True)

## 5.5 Группировка возрастов (AgeGroup)

In [ ]:
def age_group(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 12:
        return 'Child'
    elif age <= 18:
        return 'Teenager'
    elif age <= 35:
        return 'Young Adult'
    elif age <= 60:
        return 'Adult'
    else:
        return 'Senior'

train_df['AgeGroup'] = train_df['Age'].apply(age_group)
test_df['AgeGroup'] = test_df['Age'].apply(age_group)

## 5.6 Итоговая таблица новых признаков

In [ ]:
# Создаем список новых колонок, которые мы добавили
new_features = [
    'Title', 'FamilySize', 'IsAlone', 'HasCabin', 'TicketGroupSize',
    'FarePerPerson', 'AgeGroup'
]

# Показываем ключевые старые и все новые признаки
columns_to_show = ['Survived', 'Pclass', 'Sex', 'Age'] + new_features
train_df[columns_to_show].head(10).style.background_gradient(cmap='Blues') # покажем 10 строк для наглядности

# 6. Подготовка данных для моделирования

## 6.1 Кодирование категориальных признаков

In [ ]:
# Разделим колонки на типы
# Категориальные признаки (нужно закодировать)
categorical_cols = ['Sex', 'Embarked', 'Title', 'AgeGroup']

# Числовые признаки (можно оставить как есть или масштабировать позже)
numeric_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare',
                'FamilySize', 'IsAlone', 'HasCabin', 'TicketGroupSize', 'FarePerPerson']

# Признаки, которые не будем использовать в модели (идентификаторы, текст, уже обработанные)
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']

print("Категориальные признаки для кодирования:", categorical_cols)
print("Числовые признаки:", numeric_cols)
print("Исключаем:", drop_cols)

In [ ]:
# Копируем данные для моделирования
X_train = train_df.copy()
X_test = test_df.copy()

# One-Hot Encoding для категориальных признаков
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Убедимся, что колонки в train и test совпадают (добавляем недостающие в test)
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0

# Приводим порядок колонок к одинаковому
X_test = X_test[X_train.columns]

# Удаляем ненужные колонки
X_train = X_train.drop(columns=drop_cols, errors='ignore')
X_test = X_test.drop(columns=drop_cols, errors='ignore')

# Целевая переменная
y_train = X_train['Survived']
X_train = X_train.drop('Survived', axis=1)

print(f"Размер X_train: {X_train.shape}")
print(f"Размер X_test: {X_test.shape}")

In [ ]:
# Посмотрим первые строки после кодирования
X_train.head()

In [ ]:

#  Проверка пропусков (используем утилиты)
display_side_by_side(
    missing_data(X_train),
    missing_data(X_test),
    "Пропуски TRAIN",
    "Пропуски TEST"
)

 # Сравнение пропусков между train и test ( утилита)
print("\n СРАВНЕНИЕ ПРОПУСКОВ TRAIN vs TEST:")
display(compare_train_test(X_train, X_test))

#  Проверка колонок (совпадают ли train и test)
print("\n ПРОВЕРКА КОЛОНОК:")
print(f"   Колонок в X_train: {len(X_train.columns)}")
print(f"   Колонок в X_test:  {len(X_test.columns)}")

if set(X_train.columns) == set(X_test.columns):
    print(" Колонки X_train и X_test совпадают!")
else:
    print(" Колонки НЕ совпадают! Различия:")
    only_in_train = set(X_train.columns) - set(X_test.columns)
    only_in_test = set(X_test.columns) - set(X_train.columns)
    if only_in_train:
        print(f"      Только в X_train: {only_in_train}")
    if only_in_test:
        print(f"      Только в X_test: {only_in_test}")

#  Баланс классов
print("\n БАЛАНС КЛАССОВ (целевая переменная):")
survived_counts = y_train.value_counts()
print(f"   Выжили (1):   {survived_counts[1]} ({survived_counts[1]/len(y_train)*100:.1f}%)")
print(f"   Погибли (0):  {survived_counts[0]} ({survived_counts[0]/len(y_train)*100:.1f}%)")

## 6.2 Масштабирование числовых признаков

In [ ]:
# Определяем числовые колонки для масштабирования
# (исключаем бинарные признаки, которые уже 0/1)
numeric_features_for_scaling = [
    'Age', 'Fare', 'FamilySize', 'TicketGroupSize', 'FarePerPerson'
]

# Бинарные признаки (их масштабировать не нужно)
binary_features = ['IsAlone', 'HasCabin']

# Создаем scaler и обучаем на X_train
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Масштабируем числовые признаки
X_train_scaled[numeric_features_for_scaling] = scaler.fit_transform(X_train[numeric_features_for_scaling])
X_test_scaled[numeric_features_for_scaling] = scaler.transform(X_test[numeric_features_for_scaling])

print("\n Масштабирование выполнено!")

# 7. Построение и оценка моделей

## 7.1 Baseline модель

**Разделение на обучающую и валидационную выборки**

In [ ]:
# Используем масштабированные данные
X = X_train_scaled
y = y_train

# Разделяем: 80% обучение, 20% валидация
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y  # сохраняем пропорцию выживших
)

print("="*50)
print("РАЗДЕЛЕНИЕ ДАННЫХ")
print("="*50)
print(f"Обучающая выборка:  {X_train_part.shape[0]} записей ({X_train_part.shape[0]/len(X)*100:.0f}%)")
print(f"Валидационная:      {X_val.shape[0]} записей ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"\nБаланс классов в обучающей выборке:")
print(f"  Выживших:  {y_train_part.sum()} ({y_train_part.mean()*100:.1f}%)")
print(f"  Погибших:  {(~y_train_part.astype(bool)).sum()} ({(1-y_train_part.mean())*100:.1f}%)")
print(f"\nБаланс классов в валидационной выборке:")
print(f"  Выживших:  {y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"  Погибших:  {(~y_val.astype(bool)).sum()} ({(1-y_val.mean())*100:.1f}%)")

## 7.2 Создание и сравнение моделей

In [ ]:
# Создание и сравнение моделей (без настройки гиперпараметров)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import time

# Список моделей для сравнения
models = {
    'Baseline (все погибли)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_SEED, n_estimators=100),
    'SVM': SVC(random_state=RANDOM_SEED, probability=True)
}

print("="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ (БЕЗ НАСТРОЙКИ)")
print("="*60)

results = []

for name, model in models.items():
    start_time = time.time()

    # Обучение
    model.fit(X_train_part, y_train_part)

    # Предсказания
    y_train_pred = model.predict(X_train_part)
    y_val_pred = model.predict(X_val)

    # Метрики
    train_acc = accuracy_score(y_train_part, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    train_time = time.time() - start_time

    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Val Accuracy': val_acc,
        'Overfitting': train_acc - val_acc,
        'Time (sec)': train_time
    })

    print(f"\n{name}:")
    print(f"  Train Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"  Val Accuracy:   {val_acc:.4f} ({val_acc*100:.2f}%)")
    print(f"  Overfitting:    {train_acc - val_acc:.4f}")
    print(f"  Time:           {train_time:.2f} сек")

# Сводная таблица
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Val Accuracy', ascending=False)

print("\n" + "="*60)
print("СВОДНАЯ ТАБЛИЦА (сортировка по Val Accuracy)")
print("="*60)
display(results_df)

### 7.2.1 Подбор гиперпараметров (GridSearchCV)

In [ ]:
# Только для Random Forest (самый потенциал улучшения)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
                       param_grid, cv=5, scoring='accuracy')
rf_grid.fit(X_train_part, y_train_part)
print(f"Best RF: {rf_grid.best_score_:.4f} with {rf_grid.best_params_}")

### Вывод:
Несмотря на подбор гиперпараметров, Random Forest не смог обогнать LR:
- Logistic Regression: 83.80% (без настройки!)
- Tuned Random Forest: 83.15%
- LR все равно лучше на 0.65%

## 7.3 Анализ лучшей модели

In [ ]:
# Берем лучшую модель из сравнения
best_model = models['Logistic Regression']

# Детальный анализ на валидационной выборке
y_val_pred = best_model.predict(X_val)

print("="*60)
print("ДЕТАЛЬНЫЙ АНАЛИЗ ЛУЧШЕЙ МОДЕЛИ: Logistic Regression")
print("="*60)

# Матрица ошибок
print("\nМАТРИЦА ОШИБОК:")
cm = confusion_matrix(y_val, y_val_pred)
print(pd.DataFrame(cm, index=['Факт: Погиб', 'Факт: Выжил'],
                   columns=['Предсказано: Погиб', 'Предсказано: Выжил']))

# Classification report
print("\nКЛАССИФИКАЦИОННЫЙ ОТЧЕТ:")
print(classification_report(y_val, y_val_pred,
                          target_names=['Погиб (0)', 'Выжил (1)']))

# ROC-AUC
from sklearn.metrics import roc_auc_score, roc_curve
y_val_proba = best_model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_val_proba)
print(f"\nROC-AUC: {roc_auc:.4f}")

### 7.3.1 График ROC-кривой

In [ ]:
# ROC-кривая
fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Случайная модель')
plt.xlabel('False Positive Rate (1 - Специфичность)', fontsize=12)
plt.ylabel('True Positive Rate (Чувствительность)', fontsize=12)
plt.title('ROC-кривая', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7.3.2 Важность признаков (коэффициенты)

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': best_model.coef_[0]
}).sort_values('coefficient', ascending=False)

# Визуализация топ-10
plt.figure(figsize=(10, 8))
colors = [COLORS['survived'] if coef > 0 else COLORS['died']
          for coef in feature_importance.head(10)['coefficient']]
plt.barh(feature_importance.head(10)['feature'],
         feature_importance.head(10)['coefficient'],
         color=colors)
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.xlabel('Коэффициент (влияние на выживание)', fontsize=12)
plt.title('Топ-10 важных признаков (Logistic Regression)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("ИНТЕРПРЕТАЦИЯ КОЭФФИЦИЕНТОВ")
print("="*60)
print(" Положительный коэффициент → повышает шанс выживания")
print(" Отрицательный коэффициент → снижает шанс выживания")

# 8. Интерпретация и выводы

In [ ]:
def color_coef(val):
    """Цвет для коэффициентов: зеленый (+) и красный (-)"""
    if val > 0:
        return f'color: {COLORS["survived"]}; font-weight: bold'
    elif val < 0:
        return f'color: {COLORS["died"]}; font-weight: bold'
    else:
        return 'color: gray'

# Топ-15 признаков
styled_df = feature_importance.head(15).style.format({'coefficient': '{:+.4f}'})
styled_df = styled_df.applymap(color_coef, subset=['coefficient'])
styled_df

## 8.1 Ключевые факторы выживания

На основе анализа коэффициентов логистической регрессии (лучшей модели с точностью 83.8%) можно выделить **три главных фактора**, определявших выживание на "Титанике":

#### ✅ Факторы, повышающие шансы на выживание:

| Фактор | Коэффициент | Интерпретация |
|--------|-------------|---------------|
| **Наличие каюты (HasCabin)** | **+0.91** | Самый сильный положительный фактор. Пассажиры с каютой имели значительно больше шансов выжить |
| **Титул "Mrs"** | **+0.73** | Замужние женщины выживали чаще (связано с приоритетом "женщины и дети") |
| **Дети (AgeGroup_Child)** | **+0.59** | Дети до 12 лет имели повышенные шансы на спасение |
| **Порт посадки Q (Queenstown)** | **+0.32** | Пассажиры из Queenstown выживали чаще (вероятно, связано с классом билета) |

#### ❌ Факторы, снижающие шансы на выживание:

| Фактор | Коэффициент | Интерпретация |
|--------|-------------|---------------|
| **Порт посадки S (Southampton)** | **-0.33** | Пассажиры из Southampton выживали реже (большинство пассажиров 3-го класса) |
| **Одиночки (IsAlone)** | **-0.29** | Пассажиры без семьи выживали хуже |
| **Количество родителей/детей (Parch)** | **-0.22** | Небольшое негативное влияние |
| **Титул "Miss"** | **-0.18** | Незамужние женщины выживали реже, чем "Mrs" |

## 8.2 Что подтвердилось из исторических фактов?

1. **"Женщины и дети в первую очередь"** ✅
   - Титул "Mrs" (+0.73) и "Child" (+0.59) - среди топ-3 положительных факторов
   - "Miss" имеет отрицательный коэффициент (вероятно, молодые девушки могли быть в 3-м классе)

2. **1-й класс спасался лучше** ✅
   - HasCabin (+0.91) - маркер более высокого класса
   - Pclass_1 не попал в топ-15, но HasCabin его эффективно заменяет

3. **Семья помогала выжить** ✅
   - IsAlone (-0.29) - одиночки выживали хуже
   - Семьи из 2-4 человек имели преимущество

## 8.4 Итоговая оценка модели

| Метрика | Значение |
|---------|----------|
| **Точность (Accuracy)** | 83.80% |
| **Улучшение над baseline** | +22.35% |
| **ROC-AUC** | ~0.87 |

Модель показывает **хорошую предсказательную способность** и значительно превосходит "глупую" стратегию предсказания гибели всех пассажиров (61.45%).

In [ ]:
# ============================================
# ФИНАЛЬНЫЙ САБМИТ
# ============================================

X_test_final = test_df.copy()

# One-Hot Encoding для категориальных признаков
X_test_final = pd.get_dummies(X_test_final, columns=categorical_cols, drop_first=True)

# Добавляем недостающие колонки 
for col in X_train_scaled.columns:
    if col not in X_test_final.columns:
        X_test_final[col] = 0

# Приводим порядок колонок к тому же, что в X_train_scaled
X_test_final = X_test_final[X_train_scaled.columns]

# Масштабируем числовые признаки
X_test_final[numeric_features_for_scaling] = scaler.transform(X_test_final[numeric_features_for_scaling])

# 3. Обучаем лучшую модель на ВСЕХ данных
best_model = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
best_model.fit(X_train_scaled, y_train)

# 4. Предсказания
predictions = best_model.predict(X_test_final)

# 5. Создаем сабмит
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions.astype(int)
})

submission.to_csv('submission.csv', index=False)

print(" submission.csv создан!")
print(submission.head())
print(f"\nРаспределение предсказаний:")
print(pd.Series(predictions).value_counts())
print(f"  Выживших: {(predictions == 1).sum()}")
print(f"  Погибших: {(predictions == 0).sum()}")

# 9. Ограничения и пути улучшения (Next Steps)

## 9.1 Ограничения текущего решения

| Ограничение | Описание | Влияние |
|-------------|----------|---------|
| **Дисбаланс классов** | Выживших 38.4%, погибших 61.6% | Модель может быть смещена в сторону класса "погиб" |
| **Пропуски в данных** | Age (20% пропусков), Cabin (77% пропусков) | Потеря информации, использование медианного заполнения |
| **Линейность модели** | Logistic Regression предполагает линейные связи | Не учитывает сложные нелинейные взаимодействия признаков |
| **Малый объем данных** | Всего 891 запись для обучения | Ограниченная репрезентативность, риск переобучения |
| **Отсутствие валидации на отложенной выборке** | Использовалось только train/val разбиение | Нет финальной проверки на "невидимых" данных |

## 9.2 Пути улучшения

- Балансировка классов (SMOTE или классовые веса)
- Заполнение Age через IterativeImputer
- Создание признака DeckPriority (A-F)
- Переход на XGBoost или CatBoost
- Кросс-валидация (5-10 фолдов)
- Настройка порога классификации (не 0.5)